# Train the Phase 1 ECG denoising model on Kaggle

This notebook clones the project, downloads the LUDB and MIT-BIH Noise Stress Test databases, preprocesses the signals, and trains the model. Before running it, enable a **GPU** and **Internet** in the Kaggle notebook settings. Files written below `/kaggle/working` are included when you save a notebook version.

## 1. Set up the repository

This notebook assumes the repository is public. Enable Internet in Kaggle settings before running it.


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import torch

KAGGLE_WORKING = Path('/kaggle/working')
REPO_DIR = KAGGLE_WORKING / 'phase1'
REPO_URL = 'https://github.com/vzyhug/phase1.git'
REPO_BRANCH = 'main'

if not KAGGLE_WORKING.exists():
    raise RuntimeError('This notebook is intended to run in a Kaggle notebook session.')

if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('WARNING: No GPU is enabled. Select a GPU accelerator in Kaggle settings.')


In [ ]:
if (REPO_DIR / ".git").is_dir():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', REPO_BRANCH, '--depth', '1'], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'reset', '--hard', f'origin/{REPO_BRANCH}'], check=True)
else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print(f'Repository ready at {REPO_DIR}')
subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], check=True)


In [ ]:
%pip install -q -r requirements.txt

## 2. Download and validate the training data

The archives contain nested directories. This extraction step deliberately flattens the WFDB files into the paths expected by the preprocessing code and verifies the required records before continuing.

In [ ]:
from urllib.request import Request, urlopen
from zipfile import ZipFile
from tqdm.auto import tqdm

DOWNLOAD_DIR = KAGGLE_WORKING / 'downloads'
LUDB_DIR = REPO_DIR / 'data/raw/ludb_database'
NST_DIR = REPO_DIR / 'data/raw/mit_bih_nst'
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

datasets = {
    'ludb': (
        'https://physionet.org/static/published-projects/ludb/'
        'lobachevsky-university-electrocardiography-database-1.0.1.zip',
        DOWNLOAD_DIR / 'ludb.zip',
        LUDB_DIR,
    ),
    'nst': (
        'https://physionet.org/static/published-projects/nstdb/'
        'mit-bih-noise-stress-test-database-1.0.0.zip',
        DOWNLOAD_DIR / 'nst.zip',
        NST_DIR,
    ),
}

def download(url, destination):
    if destination.exists() and destination.stat().st_size > 0:
        print(f'Using cached {destination.name}')
        return
    print(f'Downloading {destination.name} ...')
    partial = destination.with_suffix(destination.suffix + '.part')
    partial.unlink(missing_ok=True)
    try:
        request = Request(url, headers={'User-Agent': 'Kaggle phase1 training notebook'})
        with urlopen(request, timeout=60) as response, partial.open('wb') as output:
            total = int(response.headers.get('Content-Length', 0)) or None
            with tqdm(total=total, unit='B', unit_scale=True, desc=destination.name) as progress:
                while chunk := response.read(1024 * 1024):
                    output.write(chunk)
                    progress.update(len(chunk))
        partial.replace(destination)
    except Exception:
        partial.unlink(missing_ok=True)
        raise

def extract_flat(archive, destination):
    destination.mkdir(parents=True, exist_ok=True)
    with ZipFile(archive) as zip_file:
        members = [member for member in zip_file.infolist() if not member.is_dir()]
        for member in members:
            target = destination / Path(member.filename).name
            if not target.name or (target.exists() and target.stat().st_size == member.file_size):
                continue
            with zip_file.open(member) as source, target.open('wb') as output:
                shutil.copyfileobj(source, output)

for _, (url, archive, destination) in datasets.items():
    download(url, archive)
    extract_flat(archive, destination)

ludb_headers = list(LUDB_DIR.glob('*.hea'))
missing_nst = [
    f'{record}.{extension}'
    for record in ('bw', 'em', 'ma')
    for extension in ('hea', 'dat')
    if not (NST_DIR / f'{record}.{extension}').is_file()
]
if not ludb_headers:
    raise FileNotFoundError(f'No LUDB header files were extracted to {LUDB_DIR}')
if missing_nst:
    raise FileNotFoundError(f'Missing MIT-BIH NST files: {missing_nst}')

print(f'Data ready: {len(ludb_headers)} LUDB records and 3 NST noise records')

## 3. Preprocess the ECG data

This resamples, segments, normalizes, and synthesizes noisy ECG samples. Rerunning this command rebuilds the generated arrays.

In [ ]:
!python run_workflow.py --mode preprocess

## 4. Train the model

The default command trains the 1D U-Net using `configs/base.yaml`. Change `MODEL_CHOICE` to `'2'` to train the original ConditionalModel implementation. Checkpoints and the CSV training log are written to `/kaggle/working/phase1/checkpoints`.

In [ ]:
MODEL_CHOICE = '1'
!python run_workflow.py --mode train --model {MODEL_CHOICE}

In [ ]:
checkpoint_archive = shutil.make_archive(
    str(KAGGLE_WORKING / 'phase1_checkpoints'),
    'zip',
    root_dir=REPO_DIR,
    base_dir='checkpoints',
)
print(f'Checkpoint archive: {checkpoint_archive}')

## 5. Run inference and evaluation (optional)

Run these cells after training. The evaluation image is also kept under `/kaggle/working/phase1` when the notebook version is saved.

In [ ]:
!python run_workflow.py --mode infer

In [ ]:
!python run_workflow.py --mode eval

from IPython.display import Image, display
display(Image(filename=str(REPO_DIR / 'evaluation_results.png')))